In [1]:
# import packages
from Equipments import BNC575, Weeder, SR400, MCBOX, Andor, Agilent, ThorlabsPM
from Equipments.MCBOX import generate_test_voltages

from DataProcessing.Plot import PlotSaver
import numpy as np
import pyvisa
import time
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

In [2]:
ps = PlotSaver("X:/migratedData/Rydberg_QIS/data")

In [3]:
rm = pyvisa.ResourceManager()
print(rm.list_resources())

('ASRL1::INSTR', 'ASRL2::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL6::INSTR', 'ASRL7::INSTR', 'GPIB0::9::INSTR', 'TCPIP0::169.254.110.10::inst0::INSTR', 'USB0::0x1313::0x807B::241023118::0::INSTR')


In [4]:
mcbox = MCBOX.MCBOX(find_device = "USB-3114")

USB-3114 found.
Board number: 0


In [5]:
bias_channel = 11

In [42]:
mcbox.set_voltage_1chan(channel = bias_channel, voltage = 1.1, bipolar = True, display = True) # changed from 2.8 V

Channel 11 has been set to 1.1 V.


In [ ]:
agilent = Agilent.Agilent('TCPIP0::169.254.110.10::inst0::INSTR',power_limit = 0, freq_limit = 40e9)

In [ ]:
pm = ThorlabsPM.ThorlabsPM('USB0::0x1313::0x807B::241023118::0::INSTR')

In [ ]:
pm.readout(wavelength = 1064)

In [ ]:
pm.readout(wavelength = 1064, average = 50)

## agilent.set_frequency(frequency = 0.7e9)

## 700MHz

In [ ]:
agilent.set_frequency(frequency = 0.7e9)

output_power = []
RF_power_list = np.linspace(-30, 0, 31) # [dBm]
bias_list = np.linspace(-4,7,41) # [V]
bias_channel = 11  # added this
print(RF_power_list)
print(bias_list)

agilent.set_power(RF_power_list[0])
agilent.start_output()

for RF_power in RF_power_list:
    print(f"Now checking {RF_power}dBm")
    agilent.set_power(RF_power)
    time.sleep(1)
    sub_output_power = []
    for bias in bias_list:
        mcbox.set_voltage_1chan(channel = bias_channel, voltage = bias, bipolar = True, display = False)
        time.sleep(2e-3)
        power = pm.readout(wavelength = 1064,average=50) # changed from 780 nm
        sub_output_power.append(power)
    print(sub_output_power)
    output_power.append(sub_output_power)   
agilent.stop_output()
mcbox.set_voltage_1chan(channel = bias_channel, voltage = 0, bipolar = True, display = True)

In [ ]:
ps.Plot3D(x = RF_power_list+30, y = bias_list, Z = np.array(output_power),
          xlabel = "RF power after Amp (dBm) @ 700MHz", ylabel = "DC bias (V)", zlabel = "Output Power (W)",transpose=True)  

## 38GHz

RF_list_top = np.linspace(-14.5,-5,50)
RF_list_bot = np.linspace(-35,-15,20)
RF_power_list = np.concatenate((RF_list_bot,RF_list_top), axis=0)
print(RF_power_list)

In [ ]:
angle = 85.0 # [deg]
#power = 1.6 # [mW]

output_power = []
#RF_power_list = np.linspace(-20,-3,39) # [dBm]
#RF_list_top = np.linspace(-14.5,-5,50)
#RF_list_bot = np.linspace(-35,-15,20)
#RF_power_list = np.concatenate((RF_list_bot,RF_list_top), axis=0)

#bias_list = np.linspace(-10,10,41) # [V]

RF_power_list = np.arange(-20,-2.6,0.4)
bias_list = np.arange(-10,10.25,0.25)

bias_channel = 11  # added this
print(RF_power_list)
print(bias_list)

agilent.set_frequency(frequency = 38.0e9)  # changed from 10 GHz
agilent.set_power(RF_power_list[0])
agilent.start_output()

for RF_power in RF_power_list:
    print(f"Now checking {RF_power} dBm")
    agilent.set_power(RF_power)
    time.sleep(0.1)
    sub_output_power = []
    for bias in bias_list:
        mcbox.set_voltage_1chan(channel = bias_channel, voltage = bias, bipolar = True, display = False)
        time.sleep(2e-3)
        power = pm.readout(wavelength = 1064, average = 50) # changed from 780 nm
        sub_output_power.append(power)
    print(sub_output_power)
    output_power.append(sub_output_power)   
agilent.stop_output()
mcbox.set_voltage_1chan(channel = bias_channel, voltage = 0, bipolar = True, display = True)

In [ ]:
RF_power_list += 30  # accounts for the amplifier
RF_power_list = (1/1000)*10**(RF_power_list/10)  # convert from dBm to W

In [ ]:
ps.Plot3D(x = RF_power_list, y = bias_list, Z = np.array(output_power)*1e+3,
          xlabel = "RF power (W) @ 26.5 GHz", ylabel = "DC bias (V)", zlabel = "Output power (mW)", transpose = True, notes = {"angle (deg)": angle})  

In [ ]:
pm.close()

In [ ]:
agilent.close()

In [20]:
mcbox.set_voltage_1chan(channel = bias_channel, voltage = 0, bipolar = True, display = True)

Channel 11 has been set to 0 V.


In [ ]:
rm.list_opened_resources()

In [ ]:
rm.close()

## Check the polarization

In [ ]:
def func(x,a,b,c,d):
    return a*np.sin(b*x+c)+d

In [ ]:
bias_list = np.linspace(-10,10,251) # [V]
transmission = []

for bias in bias_list:
    mcbox.set_voltage_1chan(channel = bias_channel, voltage = bias, bipolar = True, display = False)
    power = pm.readout(wavelength = 1064, average = 50)
    transmission.append(power)

In [ ]:
initial_guess = [0.002,2.5,0,0]

popt1, pcov1 = curve_fit(
            func, bias_list, transmission, p0 = initial_guess, maxfev = 10000000, method ='lm')

# popt1, pcov1 = curve_fit(
#             func, bias_list, transmission,maxfev=10000000,method='lm'
#         )

plt.figure()
plt.scatter(bias_list,np.array(transmission)*1e+03, c = "r", label = "measurement")
plt.plot(bias_list,func(bias_list,*popt1)*1e+03, label = "fitted curve")
plt.xlabel('bias voltage (V)')
plt.ylabel("output power (mW)")
plt.title(f"DC V_pi = {round(np.pi/popt1[1],3)} V, HWP @ {angle} deg.")
plt.legend()
plt.show()

In [ ]:
ps.Plot2D(x = bias_list, y = None, Z = transmission, xlabel = 'DC bias (V)', ylabel = 'Y', zlabel = 'Output power (W)',notes = {"angle (deg)": angle})

In [ ]:
print(len(bias_list))
print(len(transmission))

## test the degrade for RF

In [ ]:
plt.scatter(voltage,transimission_intensity)

In [ ]:
transimission_intensity = np.array([5.8,])
voltage = np.array([-4,-3.5,-3,-2.5,-2,-1.5,-1,-0.5,0,])

In [ ]:
plt.scatter(voltage,transimission_intensity)

In [ ]:
def func(x,a,b,c):
    return a*np.sin(b*x)+c

popt1, pcov1 = curve_fit(
            func, voltage, transimission_intensity,maxfev=100000,
        )

In [ ]:
popt1

In [ ]:
plt.plot(voltage, func(voltage,*popt1))